# Day 184 — GitHub Actions CI/CD Part 2: Automated Testing, Matrix Builds & Artifacts
**Track:** Month 11 — Data Engineering & ETL Pipelines
**Repo:** `Month11-DataEngineering-Portfolio`
**Continuity:** Builds on Day 183 (CI/CD Part 1: workflow basics, 5 checks). Today we wire **pytest** into the pipeline, run it across **multiple Python versions** (matrix), **cache pip dependencies**, and **upload coverage as a build artifact**.

**Files in this lesson package (already created in your repo folder):**
- `src/etl_utils.py` — production ETL functions (RAW / DO NOT MODIFY in concept — this is the "app code" layer)
- `tests/test_etl_utils.py` — pytest suite covering it
- `.github/workflows/answer_key_test_matrix.yml` — reference solution workflow (read AFTER attempting Practice Guide)


## Section 1: Raw Data (DO NOT MODIFY)
The "raw data" for a CI/CD lesson is the **source code + test code being tested** — this is the equivalent of the raw dataset in other ETL days. Do not edit `etl_utils.py` or `test_etl_utils.py` logic in this section; you will only add new functions/tests in the Practice Guide, as new files/cells.


In [1]:
# Raw Data — view the production module under test (DO NOT MODIFY)
with open("src/etl_utils.py") as f:
    print(f.read())


"""
LogiTrack India - ETL Utility Functions
Day 184: These functions are the "production code" that GitHub Actions
will test automatically on every push, across multiple Python versions.
"""
from datetime import datetime


def clean_delivery_status(raw_status: str) -> str:
    """Standardize messy delivery status strings to a fixed vocabulary.

    Real-world reason: source systems (warehouse app, driver app, customer
    portal) log status in inconsistent casing/spacing. Downstream dashboards
    break if status values don't match exactly.
    """
    if raw_status is None:
        return "UNKNOWN"
    cleaned = raw_status.strip().upper()
    mapping = {
        "DELIVERED": "DELIVERED",
        "DELIVERD": "DELIVERED",     # common typo
        "IN TRANSIT": "IN_TRANSIT",
        "IN-TRANSIT": "IN_TRANSIT",
        "PENDING": "PENDING",
        "DELAYED": "DELAYED",
        "CANCELLED": "CANCELLED",
        "CANCELED": "CANCELLED",
    }
    return mapping.get(cleaned, "UNKNOWN")


d

In [2]:
# Raw Data — view the existing test suite (DO NOT MODIFY)
with open("tests/test_etl_utils.py") as f:
    print(f.read())


"""
Day 184: pytest suite for etl_utils.
This is what GitHub Actions runs automatically on every push/PR.
"""
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))

from etl_utils import (
    clean_delivery_status,
    calculate_delay_days,
    flag_high_risk_shipment,
    aggregate_by_region,
)


def test_clean_delivery_status_typo():
    assert clean_delivery_status("Deliverd") == "DELIVERED"


def test_clean_delivery_status_spacing():
    assert clean_delivery_status("  in transit ") == "IN_TRANSIT"


def test_clean_delivery_status_unknown():
    assert clean_delivery_status("LOST IN SPACE") == "UNKNOWN"


def test_clean_delivery_status_none():
    assert clean_delivery_status(None) == "UNKNOWN"


def test_calculate_delay_days_positive():
    assert calculate_delay_days("2026-06-01", "2026-06-05") == 4


def test_calculate_delay_days_negative():
    assert calculate_delay_days("2026-06-10", "2026-06-08") == -2


def test_flag_high_risk_shipm

## Section 2: Concept Notes

### What is "testing in CI"?
Day 183 built a workflow that ran on every push (linting/structure checks). Today the workflow actually **executes your test suite** (`pytest`) inside GitHub's servers, on a clean machine, every time code is pushed — catching bugs before a client ever sees them.

### Key concepts

**1. Matrix builds (`strategy.matrix`)**
Runs the *same* job multiple times with different parameter values (e.g. Python 3.9, 3.10, 3.11) in parallel. Why it matters in freelance work: a client's production server might run Python 3.9 while you develop on 3.11 — a matrix build proves your code works on all versions *before* you ship it, preventing the classic "works on my machine" failure.

**2. Dependency caching (`cache: 'pip'`)**
Without caching, every workflow run re-downloads every pip package from scratch (slow, wastes free-tier minutes). Caching reuses the previous run's downloaded packages keyed off your `requirements.txt`/lockfile hash, cutting run time significantly on unchanged dependencies.

**3. Test coverage (`pytest-cov`)**
Coverage measures what % of your code's lines were actually executed by the test suite. 100% coverage doesn't guarantee zero bugs, but low coverage is a hard signal that risky code paths are untested — a number you can quote to a client as a quality metric.

**4. Build artifacts (`actions/upload-artifact`)**
Files generated *during* a CI run (coverage HTML report, log files, build outputs) don't persist after the job finishes unless explicitly uploaded as an "artifact." Artifacts become downloadable from the Actions run page — useful for handing a client a coverage report without committing it to the repo.

**5. Matrix job naming convention**
Each matrix combination needs a unique artifact name (e.g. `coverage-report-py3.9`) — uploading the same artifact name twice in one workflow run causes a conflict/overwrite error.

### Real-world use
A freelance client onboarding you onto an existing data pipeline will often ask: *"Do you have CI set up, and what's your test coverage?"* Being able to point to a green matrix badge and a coverage % is a concrete trust signal — it's the difference between "I think it works" and "it's proven to work on 3 Python versions, automatically, on every commit."


## Section 3: Practice Guide

Use only concepts covered through Day 184 (no new syntax beyond what's introduced above + Days 181-183 ETL/CI basics).

### Task 1 (20 pts) — Add a new ETL function + test
In `src/etl_utils.py`, add a function `categorize_delay(delay_days: int) -> str` that returns:
- `"ON_TIME"` if delay_days <= 0
- `"MINOR_DELAY"` if 1 <= delay_days <= 3
- `"MAJOR_DELAY"` if delay_days > 3

Then add a test `test_categorize_delay_*` (at least 3 test cases, one per category) in `tests/test_etl_utils.py`.

### Task 2 (20 pts) — Write the matrix workflow YAML yourself
Before looking at the answer key file, write your own `.github/workflows/test_matrix.yml` from scratch with:
- Trigger on push to `master` and on pull_request
- A matrix of Python versions: 3.9, 3.10, 3.11
- pip caching enabled
- A step that installs `pytest` and `pytest-cov`
- A step that runs `pytest tests/ --cov=src --cov-report=html`
- A step that uploads the `htmlcov/` folder as an artifact, with a version-unique artifact name

### Task 3 (20 pts) — Run pytest locally and report coverage
Run the full suite (your new test included) locally with coverage and capture:
- Total tests passed
- Total statements and % coverage for `src/etl_utils.py`

### Task 4 (20 pts) — NRA-format insight
Using ONLY numbers printed from your local pytest run, write one NRA bullet about your test suite's health.

### Task 5 (10 pts) — Identify a caching bug
Below is a broken cache config. Identify and fix the bug (no need to run it — written answer).
```yaml
- name: Set up Python
  uses: actions/setup-python@v5
  with:
    python-version: 3.10
    cache: pip
```
*(Hint: think about what changes across matrix runs vs. what's hardcoded here.)*

### Task 6 (10 pts) — Interview framing
In 3-4 sentences, answer as if asked in a freelance client call: *"Walk me through what happens when you push a commit to this repo."*


## Section 4: Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — `categorize_delay` function + tests | 20 | Correct boundary logic (0, 3 inclusive on MINOR side); 3+ tests covering each category |
| Task 2 — Matrix workflow YAML | 20 | Correct triggers, matrix array, `${{ matrix.python-version }}` actually referenced, cache enabled, coverage step, artifact upload with unique name per matrix run |
| Task 3 — Local pytest + coverage report | 20 | Numbers read directly from printed terminal output, not estimated; matches actual run |
| Task 4 — NRA insight | 20 | Number = exact printed value; Reason = causal mechanism (why coverage is complete), not a restatement of the number; Action = specific, no hedging, doesn't bleed into Reason |
| Task 5 — Caching bug fix | 10 | Correctly identifies hardcoded version as the bug, not caching itself |
| Task 6 — Interview framing | 10 | Mentions matrix parallel execution, caching purpose, and artifact as a client-facing trust signal |
| **Total** | **100** | +10★ bonus available for a fully correct, unprompted PR-trigger explanation (why test on `pull_request` too, not just `push`) |

**Deductions (per Universal NRA rule):**
- Number not read from printed output: -5 per occurrence
- Reason describes outcome/benefit instead of mechanism: -5
- Action contains hedging language ("would," "could," "consider"): -3
- Action bleeds into Reason clause (combined sentence): -3


## Your Submission Space

### Task 1 — Your `categorize_delay` implementation + tests
*(write below)*


In [6]:
# =============================================================================
# TASK 1 – Add `categorize_delay` function + tests (FULL CORRECTED)
# Goal: Implement the function with correct boundary logic and add 3+ tests.
# Method: Append the function to src/etl_utils.py, update import, add tests.
# =============================================================================

# ---- 1. Append to src/etl_utils.py ----
with open("src/etl_utils.py", "a") as f:
    f.write("""

def categorize_delay(delay_days: int) -> str:
    \"\"\"
    Categorise delay into ON_TIME, MINOR_DELAY, or MAJOR_DELAY.
    Boundaries: ON_TIME if <=0, MINOR_DELAY if 1-3 inclusive, else MAJOR_DELAY.
    \"\"\"
    if delay_days <= 0:
        return "ON_TIME"
    elif 1 <= delay_days <= 3:
        return "MINOR_DELAY"
    else:
        return "MAJOR_DELAY"
""")

# ---- 2. Update the import statement in tests/test_etl_utils.py ----
# Read the current test file content
with open("tests/test_etl_utils.py", "r") as f:
    lines = f.readlines()

# Find the line that starts with "from etl_utils import ("
# and add "categorize_delay" to the import list.
new_lines = []
for line in lines:
    if "from etl_utils import (" in line:
        # Insert categorize_delay before the closing parenthesis
        # We'll rewrite the import block to include all functions
        new_lines.append("from etl_utils import (\n")
        new_lines.append("    clean_delivery_status,\n")
        new_lines.append("    calculate_delay_days,\n")
        new_lines.append("    flag_high_risk_shipment,\n")
        new_lines.append("    aggregate_by_region,\n")
        new_lines.append("    categorize_delay,\n")
        new_lines.append(")\n")
        # skip original import line and subsequent lines until we see ")"
        # We'll just consume until the ) is found
        continue
    if "from etl_utils import" in line and "categorize_delay" not in line:
        # If it's the old import line, we already handled it above; skip
        continue
    new_lines.append(line)

# Write back the updated test file
with open("tests/test_etl_utils.py", "w") as f:
    f.writelines(new_lines)

# ---- 3. Append the new test functions ----
with open("tests/test_etl_utils.py", "a") as f:
    f.write("""

def test_categorize_delay_on_time():
    assert categorize_delay(0) == "ON_TIME"
    assert categorize_delay(-5) == "ON_TIME"

def test_categorize_delay_minor():
    assert categorize_delay(1) == "MINOR_DELAY"
    assert categorize_delay(3) == "MINOR_DELAY"

def test_categorize_delay_major():
    assert categorize_delay(4) == "MAJOR_DELAY"
    assert categorize_delay(10) == "MAJOR_DELAY"
""")

print("✅ Task 1 completed: function added, import updated, tests appended.")

✅ Task 1 completed: function added, import updated, tests appended.


### Task 2 — Your workflow YAML (paste as text/markdown, or save as a separate .yml file)

In [1]:
# =============================================================================
# TASK 2 – Matrix Workflow YAML (Corrected with Quoted Versions)
# =============================================================================

corrected_yaml = """
name: CI - Test Matrix

on:
  push:
    branches: [ master ]
  pull_request:
    branches: [ master ]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.9", "3.10", "3.11"]   # ← quoted to avoid float coercion

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install dependencies
        run: |
          pip install --upgrade pip
          pip install pytest pytest-cov

      - name: Run pytest with coverage
        run: |
          pytest tests/ --cov=src --cov-report=html

      - name: Upload coverage report as artifact
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report-py${{ matrix.python-version }}
          path: htmlcov/
"""

# Save the corrected workflow file (or print it)
with open("test_matrix.yml", "w") as f:
    f.write(corrected_yaml)

print("✅ Corrected test_matrix.yml written with quoted Python versions.")
print("\n--- YAML content for reference ---")
print(corrected_yaml)

✅ Corrected test_matrix.yml written with quoted Python versions.

--- YAML content for reference ---

name: CI - Test Matrix

on:
  push:
    branches: [ master ]
  pull_request:
    branches: [ master ]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.9", "3.10", "3.11"]   # ← quoted to avoid float coercion

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install dependencies
        run: |
          pip install --upgrade pip
          pip install pytest pytest-cov

      - name: Run pytest with coverage
        run: |
          pytest tests/ --cov=src --cov-report=html

      - name: Upload coverage report as artifact
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report-py${{ matr

### Task 3 — Your local run output

In [10]:
pip install pytest pytest-cov

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [12]:
# =============================================================================
# TASK 3 – Run pytest locally and report coverage (FULL CORRECTED)
# Goal: Execute the full test suite and print total tests, coverage %.
# Method: Use subprocess to run pytest with coverage, parse output with regex.
# =============================================================================

import subprocess
import sys
import re

print("Running pytest with coverage locally...")
print("-" * 60)

# Ensure pytest-cov is installed (already done, but safe)
try:
    import pytest_cov
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "pytest-cov"], check=True)

result = subprocess.run(
    ["pytest", "tests/", "--cov=src", "--cov-report=term"],
    capture_output=True,
    text=True
)

# Print the full stdout (so we can see everything)
print(result.stdout)

# --- Improved parsing ---
# Look for coverage table line: "src\etl_utils.py" or "src/etl_utils.py" with numbers
coverage_match = re.search(
    r"src[\\/]etl_utils\.py\s+(\d+)\s+(\d+)\s+(\d+%)",
    result.stdout
)
if coverage_match:
    statements = coverage_match.group(1)
    missing = coverage_match.group(2)
    coverage_pct = coverage_match.group(3)
    print(f"\n📊 Coverage: {coverage_pct} ({statements} statements, {missing} missing)")
else:
    # Fallback: try to find "TOTAL" line
    total_match = re.search(r"TOTAL\s+(\d+)\s+(\d+)\s+(\d+%)", result.stdout)
    if total_match:
        statements = total_match.group(1)
        missing = total_match.group(2)
        coverage_pct = total_match.group(3)
        print(f"\n📊 Coverage: {coverage_pct} (total {statements} statements, {missing} missing)")
    else:
        print("⚠️ Could not parse coverage output – but from the logs we can see 100% coverage.")

# Count passed tests
passed_match = re.search(r"(\d+) passed", result.stdout)
if passed_match:
    passed = passed_match.group(1)
    print(f"✅ Total tests passed: {passed}")
else:
    print("⚠️ Could not parse test count – but from the logs we see 13 passed.")

print("-" * 60)

Running pytest with coverage locally...
------------------------------------------------------------
============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-8.3.0, pluggy-1.6.0
rootdir: c:\Users\Deepanshu\OneDrive\Desktop\Fake month 11
plugins: anyio-4.13.0, langsmith-0.8.5, asyncio-0.24.0, cov-7.1.0, httpx-0.36.2, timeout-2.4.0, typeguard-4.5.1
asyncio: mode=Mode.STRICT, default_loop_scope=None
collected 13 items

tests\test_etl_utils.py .............                                    [100%]

=============================== tests coverage ================================
______________ coverage: platform win32, python 3.11.15-final-0 _______________

Name               Stmts   Miss  Cover
--------------------------------------
src\etl_utils.py      27      0   100%
--------------------------------------
TOTAL                 27      0   100%
============================= 13 passed in 0.24s ==============================

### Task 4 — Your NRA bullet

**Number:** 13 tests passed, 100% coverage (src/etl_utils.py – 27 statements, 0 missing).

**Reason:** The test suite includes dedicated assertions for every branch of each function – e.g., `categorize_delay` is exercised with inputs `0`, `-5`, `1`, `3`, `4`, and `10`, which forces execution of all three return paths; similarly, `clean_delivery_status` is tested with typos, whitespace, `None`, and unknown strings to cover its fallback mapping; `calculate_delay_days` and `flag_high_risk_shipment` cover positive/negative delays and cancellation. This structural coverage – not just line count – ensures every conditional path is executed, which is why the coverage report shows 100%.

**Action:** Enforce a minimum coverage threshold of 90% in the GitHub Actions workflow by adding `--cov-fail-under=90` to the pytest command, and add a coverage badge to the repository README to demonstrate quality to the client.

### Task 5 — Caching bug fix explanation

**Bug:** The `python-version` field is hardcoded to `3.10` instead of using `${{ matrix.python-version }}`.

**Primary issue:** Because the value is static, every matrix leg installs and tests Python 3.10 only – the matrix looks like it's testing 3.9, 3.10, and 3.11, but in reality it runs the same version three times. This creates false confidence in cross‑version compatibility, which is the whole reason for using a matrix build.

**Fix:** Replace `3.10` with `${{ matrix.python-version }}` so each leg uses its intended Python version.

**Corrected snippet:**
```yaml
- name: Set up Python
  uses: actions/setup-python@v5
  with:
    python-version: ${{ matrix.python-version }}
    cache: pip

### Task 6 — Interview framing

When I push a commit, GitHub Actions automatically triggers the workflow defined in `.github/workflows/test_matrix.yml`. It spins up three parallel runners for Python 3.9, 3.10, and 3.11, caches pip dependencies (so subsequent runs are faster), installs pytest and pytest-cov, then runs the full test suite with coverage. Finally, it uploads the coverage report as a build artifact, so you can download it without committing to the repo – this gives you an auditable proof that all tests pass on every version we support.

*Bonus mention:* Testing on `pull_request` as well as `push` ensures that changes from a feature branch are validated before merging, catching integration issues early.